# Chapter 3 — Group Comparison
**MADT6004 · Brew Lab BKK case**

Three classic tests:

| Comparison | Test |
|---|---|
| Two-group means (one continuous outcome) | t-test |
| Three or more groups (one continuous outcome) | One-way ANOVA |
| Two categorical variables | Chi-square test of independence |

Each gives a **test statistic** and a **p-value**. p < 0.05 → the groups likely differ; otherwise we can't conclude there's a difference.


## 0. Bootstrap (Colab + local)

In [ ]:
# Bootstrap — make sure brewlab.db is available, both locally and in Colab.
import os
DB_CANDIDATES = [
    "../../Integrated Data Analytics Exercise/data/brewlab.db",
    "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db",
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)
if DB_PATH is None:
    if not os.path.exists("MADT6004"):
        os.system("git clone -q https://github.com/thanachart/MADT6004.git")
    os.system("pip install -q -r 'MADT6004/Integrated Data Analytics Exercise/requirements.txt'")
    DB_PATH = "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db"
print("DB:", DB_PATH)


## 1. Setup

In [ ]:
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
from scipy import stats

conn = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])


## 2. Build a daily branch panel
Aggregate transactions to daily branch revenue, then attach branch attributes.

In [ ]:
panel = pd.read_sql("""
SELECT date(t.datetime) AS d, t.branch_id,
       SUM(t.total) AS revenue
FROM transactions t
GROUP BY t.branch_id, date(t.datetime)
""", conn)
br = pd.read_sql("SELECT branch_id, name, district, has_drive_thru FROM branches", conn)
panel = panel.merge(br, on="branch_id")
panel["d"] = pd.to_datetime(panel["d"])
panel["dow"] = panel["d"].dt.dayofweek
panel["is_weekend"] = (panel["dow"] >= 5).astype(int)
print(panel.head())


## 3. t-test — drive-thru vs walk-in revenue
Is daily branch revenue different between drive-thru and walk-in branches?

In [ ]:
a = panel.loc[panel["has_drive_thru"] == 1, "revenue"]
b = panel.loc[panel["has_drive_thru"] == 0, "revenue"]
t, p = stats.ttest_ind(a, b)
print(f"Drive-thru mean: {a.mean():.0f}  | Walk-in mean: {b.mean():.0f}")
print(f"t = {t:.3f}, p = {p:.4f}")
print("Significant at α=0.05?", p < 0.05)


## 4. ANOVA — revenue across districts
Districts have more than two levels, so use one-way ANOVA.

In [ ]:
groups = [g["revenue"].values for _, g in panel.groupby("district")]
f, p = stats.f_oneway(*groups)
print(f"ANOVA across {len(groups)} districts:  F = {f:.3f}, p = {p:.4f}")
print("Significant at α=0.05?", p < 0.05)

means = panel.groupby("district")["revenue"].mean().sort_values()
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(means.index, means.values, color="#0891B2")
ax.set_title("Mean daily revenue by district")
ax.set_ylabel("Revenue (THB)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()


## 5. Chi-square — channel choice vs branch type
Is the channel mix different at drive-thru vs walk-in branches?

In [ ]:
tx = pd.read_sql("""
SELECT t.channel, b.has_drive_thru
FROM transactions t JOIN branches b ON t.branch_id = b.branch_id
""", conn)
ct = pd.crosstab(tx["has_drive_thru"], tx["channel"])
print("Contingency table:")
print(ct)

chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f"\nχ² = {chi2:.2f}, dof = {dof}, p = {p:.4f}")
print("Significant at α=0.05?", p < 0.05)


## Discussion prompts
1. Pick one finding above and write the one-sentence business takeaway you'd give Khun Ploy.
2. The t-test on drive-thru vs walk-in compared *all* days. What pattern might it miss that a comparison conditioned on weekend/weekday could reveal?
3. Why is "p < 0.05" only one piece of the answer? What other evidence would you want before acting?
